## بوابة التدريب الإلزامية

قبل أي تنزيل أو تدريب، أضف في Colab Secrets القيم التالية: `DRPLANT_TRAINING_PROFILE`, `DRPLANT_TRAINING_PROFILE_SIGNATURE`, `DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY`. يجب أن تشير إلى profile وتوقيعه والمفتاح العام الموثوق. سيوقف الدفتر التنفيذ إذا غابت أي قيمة أو فشل فحص البيانات. لا تتجاوز هذه الخلية.


In [ ]:
import os
from google.colab import userdata
_REQUIRED_DRPLANT_GATE_SECRETS = (
    "DRPLANT_TRAINING_PROFILE",
    "DRPLANT_TRAINING_PROFILE_SIGNATURE",
    "DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY",
)
_missing = []
for _name in _REQUIRED_DRPLANT_GATE_SECRETS:
    _value = userdata.get(_name)
    if not _value:
        _missing.append(_name)
    else:
        os.environ[_name] = _value
if _missing:
    raise RuntimeError("NO-GO: configure these Colab Secrets before running any training cell: " + ", ".join(_missing))
print("Dr.Plant training gate credentials loaded; the training script will verify the signed profile and data before training.")


# Dr Plant — Lemon V3 exploratory training
Dataset DOI `10.17632/8d9fv6kpt3.3`, CC BY 4.0; 17,586 images and 18 image-level classes.
The released dataset does not expose image-level farm/tree IDs. Therefore this notebook uses a stratified image-level split and marks results exploratory/non-independent. The images were photographed individually against a plain white background, so they are not a substitute for real orchard-scene validation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone --depth 1 --branch research/morphology-v2 https://github.com/jonbander/dr-plant.git /content/dr-plant
!pip -q install torch torchvision numpy pillow scikit-learn onnx


In [ ]:
from pathlib import Path
DATA = Path('/content/drive/MyDrive/dr-plant-data/lemon-v3/Large-Scale Lemon Leaf Disease and Pest Image Data/LemonLeaf_Dataset')
OUT = Path('/content/drive/MyDrive/dr-plant-runs/lemon-v3')
assert DATA.is_dir()


In [ ]:
!python /content/dr-plant/scripts/train_lemon_v3_exploratory.py --data "$DATA" --out "$OUT" --epochs 3 --batch-size 32 --image-size 128 --unfreeze-last-blocks 0 --training-profile "$DRPLANT_TRAINING_PROFILE" --training-profile-signature "$DRPLANT_TRAINING_PROFILE_SIGNATURE" --training-data-root "$DATA" --training-trusted-public-key "$DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY"
!python /content/dr-plant/scripts/train_lemon_v3_exploratory.py --data "$DATA" --out "$OUT" --epochs 6 --batch-size 32 --image-size 128 --unfreeze-last-blocks 2 --resume --training-profile "$DRPLANT_TRAINING_PROFILE" --training-profile-signature "$DRPLANT_TRAINING_PROFILE_SIGNATURE" --training-data-root "$DATA" --training-trusted-public-key "$DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY"


## Interpretation
The held-out images are drawn from the same source collection without farm-level grouping. This score is not a field accuracy estimate. Keep this model separate from the citrus and grape tasks.